# Day 4 — Pipelines, ColumnTransformer, Feature Engineering
Reusable, leakage-free preprocessing.


In [ ]:
import pandas as pd, numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

df = sns.load_dataset('titanic')
y = df['survived']
X = df.drop(columns=['survived', 'alive', 'who', 'adult_male', 'class', 'deck', 'embark_town'])
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()
print('num:', num_cols, '\ncat:', cat_cols)


In [ ]:
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
cat_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
    ('ohe', OneHotEncoder(handle_unknown='ignore')),
])
pre = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols),
])
pipe = Pipeline([('pre', pre), ('clf', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, eval_metric='logloss', random_state=0))])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scores = cross_val_score(pipe, X, y, scoring='f1', cv=cv, n_jobs=-1)
print(f'Pipeline F1 = {scores.mean():.3f} ± {scores.std():.3f}')


## Engineered features

In [ ]:
X2 = X.copy()
X2['family_size'] = X2['sibsp'] + X2['parch'] + 1
X2['is_alone']   = (X2['family_size'] == 1).astype(int)
X2['fare_per_person'] = X2['fare'] / X2['family_size']
X2['age_bin'] = pd.cut(X2['age'], bins=[0, 12, 18, 35, 60, 120], labels=False).astype('Int64')

num_cols2 = X2.select_dtypes(include='number').columns.tolist()
cat_cols2 = X2.select_dtypes(exclude='number').columns.tolist()
pre2 = ColumnTransformer([
    ('num', num_pipe, num_cols2),
    ('cat', cat_pipe, cat_cols2),
])
pipe2 = Pipeline([('pre', pre2), ('clf', XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, eval_metric='logloss', random_state=0))])
scores2 = cross_val_score(pipe2, X2, y, scoring='f1', cv=cv, n_jobs=-1)
print(f'With engineered features F1 = {scores2.mean():.3f} ± {scores2.std():.3f}')
